# deep-acordao-tcu2 — Visão Geral

> **Mestrado em Administração Pública — Ciência de Dados e IA (IDP)**
> Autores: Bruno Aires · Candice Trigueiro · Rafael Ayoroa

Classificação automática de desfechos em acórdãos do TCU (Saúde e Educação) em três classes:
*Irregular*, *Regular com Ressalva* e *Regular*.

## Diferença para a versão anterior

Esta versão nasce **sem vazamento**:

- O `SUMARIO` é usado **apenas** como fonte de rótulo (leakage 100% impede que vire feature).
- A feature única é o **`VOTO_LIMPO`**: `VOTO` sem o dispositivo + termos de veredito residuais mascarados como `[DECISAO]`.
- Todo treino usa **pesos de classe** para regular o forte desbalanceamento (`class_weight='balanced'` no baseline, cross-entropy ponderada e Focal Loss no LegalBert-pt, `CrossEntropyLoss(weight=...)` no TextCNN).
- O escopo temporal padrão é **2016–2024** (9 anos), com a possibilidade de split temporal (treino ≤ 2022, val = 2023, teste = 2024).
- Os resultados anteriores **não** são carregados — cada notebook regenera `resultados/metricas_*.json` do zero ao ser executado.

## Ordem de execução

1. `01_pipeline_limpo.ipynb` — download 2016–2024 → rotulagem → `VOTO_LIMPO` → gate de vazamento → split → parquet.
2. `02_baseline_ponderado.ipynb` — TF-IDF + LogReg com `class_weight='balanced'` (K-Fold e hold-out temporal).
3. `03_textcnn_ponderado.ipynb` — TextCNN com `CrossEntropyLoss` ponderada (5-fold).

O notebook opcional `04_legalbert_ponderado.ipynb` está descrito em `docs/decisoes.md` — requer GPU (Colab T4).

## Guardrails desta versão

| # | Guardrail |
|---|---|
| G1 | `SUMARIO` **nunca** aparece como feature em nenhuma célula. |
| G2 | Toda feature de treino passa pelo gate `auditar_vazamento` com fração 0. |
| G3 | Todo classificador é treinado com pesos de classe explícitos. |
| G4 | `RANDOM_STATE = 42` em todo split, treino e inicialização. |
| G5 | Escopo temporal: `2016–2024`. |
| G6 | Todos os CSVs vêm do Portal de Dados Abertos do TCU — sem scraping. |

> As referências bibliográficas completas estão em `docs/referencias.md`.